<a href="https://colab.research.google.com/github/Zattyla/AIStudio-by-katty/blob/main/AIStudio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📖 Quick Start Guide

Welcome to the **Ultimate AI Cover Pipeline**. Follow these steps to create your first AI Cover:

### ⚙️ Initial Setup
1.  **Enable GPU:** Go to `Runtime` -> `Change runtime type` and select **T4 GPU**.
2.  **Run Cells 1-5:** This will prepare the environment, mount your Drive, and install all necessary tools (Applio & UVR).

### 🎤 Step-by-Step Workflow
1.  **Upload Audio:** Open the folder icon on the left, go to `inputs`, and upload your song (e.g., `mysong.mp3`).
2.  **Download Voice Model:** In **Step 6**, paste the link to your RVC model (from HuggingFace or Pixeldrain) and run it.
3.  **Separate Vocals:** In **Step 8**, type your filename and run. This will create a `Vocals` and `Instrumental` file in `outputs/uvr`.
4.  **Clean & Convert:** Run **Step 9** to remove reverb, then **Step 11** to transform the voice into your AI model.
5.  **Final Mix:** Run **Step 12** to combine the new AI Vocal with the original Instrumental.

### 📁 Where are my files?
- **Original Uploads:** `/content/inputs/`
- **Separated Tracks:** `/content/outputs/uvr/`
- **AI Vocals:** `/content/outputs/rvc/`
- **Final Result:** `/content/outputs/FINAL_COVER...mp3`

---
💡 **Pro Tip:** Use the **Table of Contents** on the left sidebar to jump between steps quickly!

In [ ]:
# @title 🧹 Environment Optimizer & Garbage Collector
import torch
import gc
import os
from IPython.display import clear_output

# @markdown ### Optimization Settings
CLEAR_LOGS = True # @param {type:"boolean"}
CLEAR_GPU_CACHE = True # @param {type:"boolean"}
DELETE_TEMPS = False # @param {type:"boolean"}

def optimize_environment():
    # 1. Clear PyTorch GPU Cache
    if CLEAR_GPU_CACHE:
        torch.cuda.empty_cache()
        gc.collect()
        print("💡 GPU Cache cleared successfully.")

    # 2. Delete Temporary hidden files (optional)
    if DELETE_TEMPS:
        temp_folders = ['/content/Applio/assets/temp', '/root/.cache']
        for folder in temp_folders:
            if os.path.exists(folder):
                shutil.rmtree(folder)
        print("🗑️ Temporary system files deleted.")

    # 3. Clear all visual logs
    if CLEAR_LOGS:
        clear_output()
        print("✨ Environment optimized! Logs cleared and GPU memory refreshed.")

optimize_environment()

In [ ]:
# @title 1. GPU Check & Environment Setup
import os
import subprocess

# Check GPU status
try:
    gpu_info = !nvidia-smi
    if "nvidia-smi" in gpu_info[0]:
        print("✅ GPU Detected:")
        print(gpu_info[0])
    else:
        print("❌ GPU not found. Go to 'Runtime' -> 'Change runtime type' and select T4 GPU.")
except:
    print("❌ Error executing nvidia-smi. Ensure Hardware Acceleration is enabled.")

# Define Workspace Paths
BASE_DIR = "/content"
DRIVE_DIR = "/content/drive/MyDrive/RVC_UVR_Project"

✅ GPU Detected:
/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# @title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in Drive if it doesn't exist
if not os.path.exists(DRIVE_DIR):
    os.makedirs(DRIVE_DIR)
    print(f"✨ Project folder created in Drive: {DRIVE_DIR}")
else:
    print(f"📂 Project folder already exists in Drive.")

In [ ]:
# @title 3. Install Dependencies
print("⏳ Installing dependencies (this may take 3-5 minutes)...")

# 1. System Updates
!apt-get update -y && apt-get install -y ffmpeg aria2

# 2. Fix NumPy versioning first to avoid conflicts
!pip install -U numpy>=2.0.0

# 3. Audio & UI Libraries
!pip install -q pydub librosa gradio==3.48.0 audio-separator[gpu]

# 4. RVC Core Dependencies
!pip install -q faiss-cpu fairseq pyworld torch torchvision torchaudio

print("✅ All dependencies installed successfully!")

In [ ]:
# @title 4. Clone Repositories & Folders
%cd /content

# Clone Applio (RVC Engine)
if not os.path.exists("Applio"):
    print("🚀 Cloning Applio...")
    !git clone https://github.com/IAHispano/Applio.git
    %cd Applio
    !pip install -r requirements.txt
else:
    print("🚀 Applio already exists.")

# Create Workspace Folders
folders = [
    "/content/inputs",
    "/content/outputs/uvr",
    "/content/outputs/rvc",
    "/content/models/uvr"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Folder structure is ready.")

In [ ]:
# @title 5. Download UVR Separation Model (Kim_Vocal_2)
import os

UVR_MODEL_PATH = "/content/models/uvr/Kim_Vocal_2.onnx"
# Public mirror links
urls = [
    "https://huggingface.co/lucasnewman/uvr-models/resolve/main/MDX-Net/Kim_Vocal_2.onnx",
    "https://huggingface.co/Eddycrack864/UVR5-ONNX/resolve/main/Kim_Vocal_2.onnx"
]

def download_model():
    if os.path.exists(UVR_MODEL_PATH) and os.path.getsize(UVR_MODEL_PATH) > 10000000:
        print("✔ UVR Model already exists and is valid.")
        return

    for url in urls:
        print(f"⏳ Downloading from: {url}")
        os.system(f'curl -L -o "{UVR_MODEL_PATH}" "{url}"')

        if os.path.exists(UVR_MODEL_PATH) and os.path.getsize(UVR_MODEL_PATH) > 10000000:
            print("✅ UVR Model downloaded successfully!")
            return

    print("❌ Failed to download model from all sources.")

download_model()

In [ ]:
# @title 6. Voice Library (Download RVC Model)
import zipfile
import shutil
import os

# @markdown ### Download Settings
# @markdown Paste the direct link to the RVC Model (HuggingFace/Pixeldrain/Drive)
MODEL_URL = "" # @param {type:"string"}
# @markdown Give a name to your model (No spaces)
MODEL_NAME = "My_AI_Voice" # @param {type:"string"}

def download_voice_model(url, name):
    if not url:
        print("⚠️ Please provide a model URL.")
        return

    model_dir = f"/content/Applio/logs/{name}"
    os.makedirs(model_dir, exist_ok=True)
    zip_path = os.path.join(model_dir, "model.zip")

    print(f"⏳ Downloading voice model: {name}...")
    # Using aria2 for high-speed download
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d {model_dir} -o "model.zip"

    if os.path.exists(zip_path):
        print(f"📦 Extracting files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(model_dir)
        os.remove(zip_path)
        print(f"✅ Model '{name}' is ready for conversion!")
    else:
        print("❌ Download failed. Check the URL.")

download_voice_model(MODEL_URL, MODEL_NAME)

In [ ]:
# @title 7. Download RVC Utilities (RMVPE & Pretraineds)
print("⏳ Downloading RVC core files (RMVPE & Base Models)...")

# Download RMVPE (High-quality Pitch Predictor)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt" -d /content/Applio/rvc/models/predictors -o rmvpe.pt

# Download Base Models (Required for stable training/inference)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/Pretraineds/f0G40k.pth" -d /content/Applio/rvc/models/pretraineds -o f0G40k.pth

print("✅ RVC Utilities downloaded successfully.")

In [ ]:
# @title 8. UVR Separation (Vocals vs Instrumental)
import os

# @markdown ### Separation Settings
# @markdown Filename of the song uploaded to 'inputs' folder (e.g., song.mp3)
AUDIO_INPUT = "song.mp3" # @param {type:"string"}
input_path = f"/content/inputs/{AUDIO_INPUT}"
output_path = "/content/outputs/uvr"

if os.path.exists(input_path):
    print(f"🎵 Splitting audio: {AUDIO_INPUT}...")

    # Using audio-separator CLI with GPU support
    # Model: Kim_Vocal_2 (Best for vocal isolation)
    !audio-separator "{input_path}" \
        --model_filename Kim_Vocal_2.onnx \
        --output_dir "{output_path}" \
        --output_format WAV \
        --normalization 0.9

    print(f"✅ Separation complete! Check files in: {output_path}")
else:
    print(f"❌ Error: {AUDIO_INPUT} not found in /content/inputs/")

In [ ]:
# @title 9. Vocal Enhancer (De-reverb & Cleaning)
import os

# @markdown ### Cleaning Settings
# @markdown Link to the De-reverb model (ONNX)
CLEAN_MODEL_URL = "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/uvr5_weights/onnx_dereverb_By_Spleeter.onnx"
CLEAN_MODEL_PATH = "/content/models/uvr/dereverb.onnx"

if not os.path.exists(CLEAN_MODEL_PATH):
    print("⏬ Downloading De-reverb model...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {CLEAN_MODEL_URL} -d /content/models/uvr -o dereverb.onnx

# @markdown Paste the Vocal filename from Step 8 (e.g., song_(Vocals)_Kim_Vocal_2.wav)
VOCAL_INPUT = "" # @param {type:"string"}
vocal_path = f"/content/outputs/uvr/{VOCAL_INPUT}"

if os.path.exists(vocal_path):
    print(f"🧹 Cleaning reverb from: {VOCAL_INPUT}...")
    !audio-separator "{vocal_path}" \
        --model_filename dereverb.onnx \
        --output_dir "/content/outputs/uvr" \
        --output_format WAV
    print("✅ Vocal cleaned and ready for RVC!")
else:
    print("❌ Error: Vocal file not found.")

In [ ]:
# @title 10 & 11. RVC Voice Conversion
import os
%cd /content/Applio

# @markdown ### Conversion Settings
# @markdown Cleaned Vocal filename (from Step 9)
INPUT_VOCAL = "" # @param {type:"string"}
# @markdown Model Name (the one you downloaded in Step 6)
MODEL_NAME = "My_AI_Voice" # @param {type:"string"}
# @markdown Pitch Shift: 0 (original), 12 (up one octave), -12 (down one octave)
PITCH = 0 # @param {type:"slider", min:-24, max:24, step:1}
# @markdown Pitch Extraction Method (RMVPE is recommended)
METHOD = "rmvpe" # @param ["rmvpe", "mangio-crepe"]

input_path = f"/content/outputs/uvr/{INPUT_VOCAL}"
output_path = f"/content/outputs/rvc/{INPUT_VOCAL}_AI.wav"

if os.path.exists(input_path):
    print(f"🎤 Converting voice to {MODEL_NAME}...")

    # Applio CLI Inference command
    !python infer.py \
        --input_path "{input_path}" \
        --output_path "{output_path}" \
        --model_name "{MODEL_NAME}" \
        --transpose {PITCH} \
        --f0_method "{METHOD}" \
        --index_rate 0.75 \
        --volume_envelope 1 \
        --protect 0.33

    print(f"🎉 SUCCESS! Your AI Vocal is ready at: {output_path}")
else:
    print("❌ Error: Input vocal not found.")

In [ ]:
# @title 12. Audio Mixing (Final Master)
from pydub import AudioSegment
import os

# @markdown ### Volume Adjustments (dB)
VOCAL_VOLUME = 0 # @param {type:"slider", min:-20, max:10, step:1}
INSTRUMENTAL_VOLUME = 0 # @param {type:"slider", min:-20, max:10, step:1}

# @markdown Instrumental filename (from Step 8)
INST_FILE = "" # @param {type:"string"}
# @markdown AI Vocal filename (from Step 11)
AI_VOCAL_FILE = "" # @param {type:"string"}

inst_path = f"/content/outputs/uvr/{INST_FILE}"
rvc_path = f"/content/outputs/rvc/{AI_VOCAL_FILE}"
final_output = f"/content/outputs/FINAL_COVER_{AI_VOCAL_FILE}.mp3"

if os.path.exists(inst_path) and os.path.exists(rvc_path):
    print("🎚️ Mixing tracks...")
    vocal = AudioSegment.from_file(rvc_path) + VOCAL_VOLUME
    instrumental = AudioSegment.from_file(inst_path) + INSTRUMENTAL_VOLUME

    # Overlay AI Vocal on top of Instrumental
    combined = instrumental.overlay(vocal)

    # Export final file
    combined.export(final_output, format="mp3")
    print(f"✨ COVER FINISHED! Download here: {final_output}")
else:
    print("❌ Error: Check your filenames for Instrumental and AI Vocal.")

In [ ]:
# @title 13. Auto-Backup & Logs Cleanup
from IPython.display import clear_output
import shutil
from datetime import datetime

# @markdown ### Backup Settings
# @markdown Zip all outputs and save to your Google Drive folder?
DO_BACKUP = True # @param {type:"boolean"}

if DO_BACKUP:
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    backup_file = f"{DRIVE_DIR}/Backup_Covers_{timestamp}"

    print(f"⏳ Creating backup in {backup_file}.zip...")
    if os.path.exists("/content/outputs"):
        shutil.make_archive(backup_file, 'zip', "/content/outputs")
        print(f"✅ Backup saved to Drive!")
    else:
        print("❌ No outputs found to backup.")

# Optional: Clear logs to keep the notebook tidy
# clear_output()
print("✨ Environment is clean and ready for the next song.")

# 🛠️ Troubleshooting & FAQ

If you encounter an error, don't panic! Check the solutions below for the most common issues in Google Colab.

### 1. 🛑 "nvidia-smi: command not found" or GPU Errors
This means the Colab instance is running on **CPU** instead of **GPU**.
* **Solution:** Go to `Runtime` -> `Change runtime type` and select **T4 GPU**. Then, restart the session and run Cell 1 again.

### 2. 🧩 "Incompatible Units" or NumPy/Dependency Conflicts
You might see red text during installation (Step 3) about `numpy` or `gradio` versions.
* **Solution:** Most of the time, you can **ignore these**. As long as the cell finishes with "✅ All dependencies installed!", the core engine will work. If the code crashes later, try `Runtime` -> `Restart session`.

### 3. 📁 "FileNotFoundError"
Python cannot find your audio or model files.
* **Solution:** Ensure your filenames have **NO SPACES** or special characters (like `ç`, `á`, `ñ`). Rename `My Song 01.mp3` to `mysong01.mp3` in the `inputs` folder.

### 4. 🤖 Robotic or Metallic Voice Quality
If the AI voice sounds distorted or unnatural:
* **Check Step 9:** Did you run the **De-reverb**? Original echo ruins the AI conversion.
* **Check Step 11:** Try changing the `METHOD` from `rmvpe` to `mangio-crepe`.
* **Index Rate:** Increase the `index_rate` to `0.8` or higher to bring back the model's original character.

### 5. 📉 Out of Memory (OOM)
The Colab RAM is full.
* **Solution:** Click `Runtime` -> `Disconnect and delete runtime` to wipe everything and start fresh. This is the "Nuclear Option" that fixes 99% of crashes.

---
✨ **Need more help?** Check the [Applio GitHub](https://github.com/IAHispano/Applio) for official documentation.